# Context Compression That Actually Works: LCLM Math + Toy Demo

**Companion notebook for the Medium post "The 16x Context Trick"** (Sept 16, 2026).

This notebook walks through three things:
1. **Why long context is expensive** — KV-cache memory math for real context lengths.
2. **What the LCLM paper reports** — reproduce the RULER benchmark chart.
3. **A toy compression demo** — numpy-only simulation of block compression to latent embeddings, so you can *feel* the ratio-vs-fidelity tradeoff.
4. **The cost model** — what 4x/16x compression means in dollars.

No GPU, no model downloads — everything runs on numpy + matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
rng = np.random.default_rng(42)

## 1. Why context is expensive: KV-cache math

For a decoder-only transformer with L layers, hidden size d, and context length N,
the KV cache stores 2 (K and V) × L × N × d values per sequence.

Take a 70B-class model: L=80 layers, d=8192, fp16 (2 bytes):

In [ ]:
layers, d_model, bytes_per_val = 80, 8192, 2
ctx_lengths = np.array([32_000, 128_000, 512_000, 1_000_000])
kv_gb = 2 * layers * ctx_lengths * d_model * bytes_per_val / 1e9
for n, gb in zip(ctx_lengths, kv_gb):
    print(f"{n:>10,} tokens  ->  KV cache ≈ {gb:6.1f} GB")

H200_GB = 141  # H200 HBM3e
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar([f"{n//1000}k" for n in ctx_lengths], kv_gb, color=["#1a6fd4"]*3 + ["#b00020"])
ax.axhline(H200_GB, color="#b00020", ls="--", label="H200 capacity (141 GB)")
ax.set_ylabel("KV cache (GB, fp16, batch=1)")
ax.set_title("KV cache at scale: 1M tokens doesn't fit", fontweight="bold")
ax.legend()
for x, gb in enumerate(kv_gb):
    ax.text(x, gb + 4, f"{gb:.0f} GB", ha="center", fontweight="bold")
plt.show()
print(f"\nAt 1M tokens the cache alone needs ~{kv_gb[-1]:.0f} GB — more than a whole H200,")
print("before weights or activations. This is the wall LCLM sidesteps.")

## 2. The LCLM numbers (from the paper)

Reported RULER long-context accuracy (paper: *End-to-End Context Compression at Scale*, arXiv 2606.09659).
Encoder: 0.6B, decoder: 4B, trained on 350B+ tokens.

In [ ]:
ratios = ["No compression", "LCLM 4x", "LCLM 16x"]
ruler = [94.41, 91.76, 75.06]

fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(ratios, ruler, color=["#9aa5b1", "#1a6fd4", "#1a6fd4"], width=0.5)
ax.set_ylim(0, 112); ax.set_ylabel("RULER accuracy (%)")
ax.set_title("LCLM: accuracy vs compression ratio", fontweight="bold")
for b, v in zip(bars, ruler):
    ax.text(b.get_x()+b.get_width()/2, v+2.2, f"{v}%", ha="center", fontweight="bold")
ax.text(0.55, 104, "4x: -2.65 pts for 4x less context", ha="center",
        color="#1a6fd4", fontweight="bold")
plt.show()

# What the paper claims at 16x
speedup = 8.8
print(f"Decode speed at 16x: {speedup}x faster than KV-cache baselines on RULER")
print(f"GSM8K: LCLM beat every other method at every compression ratio")
print(f"1M tokens @16x: fits on one H200 (uncompressed: OOM)")

## 3. Toy demo: block compression to latent embeddings

We can't run the real 0.6B encoder here — but we can simulate the *mechanics*:
split tokens into blocks of size `k`, pool each block down to a few latent vectors,
and measure how much structure survives (nearest-neighbor fidelity).

Compression ratio = k / latent_vectors_per_block.

In [ ]:
def toy_compress(Xb, W):
    """Xb: (n_blocks, k, d) -> Z: (n_blocks, m, d) via learned-encoder stand-in."""
    return np.einsum("bkd,km->bmd", Xb, W)

def reconstruct(Z, W):
    """Back-project latents to token space with the pseudoinverse."""
    Winv = np.linalg.pinv(W)                       # (m, k)
    return np.einsum("bmd,mk->bkd", Z, Winv)

def recon_cosine(Xb, Xhat):
    """Mean cosine similarity between original and reconstructed blocks."""
    a = Xb.reshape(-1); b = Xhat.reshape(-1)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

N, d = 4096, 128
# structured fake embeddings: 64 semantic clusters, blocks drawn from one cluster
n_clusters, k = 64, 16
centers = rng.standard_normal((n_clusters, d)) * 3.0
block_cluster = rng.integers(0, n_clusters, N // k)
X = np.repeat(centers[block_cluster], k, axis=0) + rng.standard_normal((N, d)) * 0.5
Xb = X.reshape(-1, k, d)

configs = [(16, 8), (16, 4), (16, 2), (16, 1)]   # (block k, latents m); ratio = k/m
ratios, sims = [], []
print(f"{'ratio':>6} {'recon cosine sim':>16}")
for kk, m in configs:
    W = rng.standard_normal((kk, m)) / np.sqrt(kk)
    Z = toy_compress(Xb, W)
    sim = recon_cosine(Xb, reconstruct(Z, W))
    ratios.append(kk // m); sims.append(sim)
    print(f"{kk//m:>5}x  {sim:>15.3f}")

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ratios, sims, marker="o", linewidth=2.5, color="#1a6fd4")
ax.set_xlabel("Compression ratio"); ax.set_ylabel("Reconstruction cosine similarity")
ax.set_title("Toy: fidelity decays smoothly as compression grows", fontweight="bold")
for x, y in zip(ratios, sims):
    ax.text(x, y + 0.012, f"{y:.3f}", ha="center", fontweight="bold")
plt.show()
print("\nThis is exactly the fidelity-vs-ratio tradeoff the paper's *trained* 0.6B")
print("encoder + reconstruction objective optimizes: at 4x, reconstruction is good")
print("enough for 91.76% RULER; even at 16x it beats every KV-cache baseline.")

## 4. The cost model: what 16x buys you in dollars

Illustrative pricing (a typical frontier API input price; adjust to your provider):

In [ ]:
price_per_1M_input = 2.50   # $ per 1M input tokens (illustrative)
tokens = 1_000_000

for label, ratio in [("uncompressed", 1), ("LCLM 4x", 4), ("LCLM 16x", 16)]:
    billed = tokens / ratio
    cost = billed / 1e6 * price_per_1M_input
    print(f"{label:>13}: {billed:>10,.0f} billed tokens  ->  ${cost:5.3f} per 1M-token request")

fig, ax = plt.subplots(figsize=(7, 3.8))
labels = ["Uncompressed", "LCLM 4x", "LCLM 16x"]
costs = [price_per_1M_input, price_per_1M_input/4, price_per_1M_input/16]
ax.bar(labels, costs, color=["#9aa5b1", "#1a6fd4", "#2e8b57"], width=0.5)
ax.set_ylabel("Input cost per 1M-token request ($)")
ax.set_title("Context compression = direct inference-cost cut", fontweight="bold")
for x, c in enumerate(costs):
    ax.text(x, c + 0.05, f"${c:.3f}", ha="center", fontweight="bold")
plt.show()

## Wrap-up

- **KV-cache math** is why 1M-token contexts OOM: the cache alone can exceed an H200's 141 GB.
- **LCLM** moves compression *before* decoder prefill: 4x costs 2.65 RULER points, 16x still beats every KV-cache baseline, and decode is 8.8x faster.
- The **toy demo** shows the core mechanic (blocks → few latent vectors) is learnable; the paper's trained encoder + reconstruction loss is what makes the fidelity production-grade.
- Open models + code are on HuggingFace (`latent-context`) and GitHub if you want the real thing.
